In [1]:
import pandas as pd

import pyspark.sql.types as T
import pyspark.sql.functions as F
from sklearn.datasets import fetch_openml

from yggdrasil.data.etl.spark.init import get_spark_session

INFO:yggdrasil.core.dataclasses:USE_VANILLA_DATACLASS: False
INFO:yggdrasil.core.dataclasses:ARBITRARY_TYPES_ALLOWED: True


In [2]:
# Initialize Spark session
spark = get_spark_session()

25/06/17 20:52:09 WARN Utils: Your hostname, MacBook-Pro-Kasan.local resolves to a loopback address: 127.0.0.1; using 192.168.1.18 instead (on interface en0)
25/06/17 20:52:09 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/06/17 20:52:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Extract raw data
pdf: pd.DataFrame = fetch_openml('mnist_784', version=1, as_frame=True).frame
pdf = pdf.rename(columns={j: str(i) for i, j in enumerate(pdf.columns[:-1])})
pdf["class"] = pd.to_numeric(pdf["class"])

schema = T.StructType(
    [
        T.StructField(i, T.IntegerType(), True)
        for i in pdf.columns
    ]
)

# Coalesce to a single partition for easier handling wide datasets
df = spark.createDataFrame(pdf, schema=schema).coalesce(1)

del pdf

In [4]:
# Mandatory transformation for the further data pre-processing
feature_cols = [str(i) for i in range(784)]
df = df.withColumn("id", F.monotonically_increasing_id()).withColumn(
    "sparse_vector",
    F.create_map(
        *list(
            sum(
                [
                    (F.lit(fid).cast("int"), F.col(f"`{fname}`").cast("double"))
                    for fid, fname in enumerate(feature_cols)
                ],
                (),
            )
        )
    )
).drop(*feature_cols).withColumnRenamed("sparse_vector", "features")

# Store the data in a dedicated warehouse
spark.sql("DROP TABLE IF EXISTS mnist_784")
df.select(
    F.col("id").cast(T.LongType()).alias("id"),
    F.col("features").cast(T.MapType(T.IntegerType(), T.FloatType())).alias("features"),
    F.col("class").cast(T.ShortType()).alias("target"),
).write.mode("overwrite").format("parquet").saveAsTable("mnist_784")

In [5]:
spark.table("mnist_784").printSchema()

root
 |-- id: long (nullable = true)
 |-- features: map (nullable = true)
 |    |-- key: integer
 |    |-- value: float (valueContainsNull = true)
 |-- target: short (nullable = true)

